In [ ]:
import csv
import os
import re
import time
import random
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

START_URL = "https://aisel.aisnet.org/misq/all_issues.html"
BASE_URL = "https://aisel.aisnet.org"
START_YEAR = 2010

OUT_FILE = os.path.join(os.getcwd(), "MISQ_Issues.csv")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0 Safari/537.36"
    )
}

ISSUE_RE = re.compile(r"/misq/vol(\d+)/iss(\d+)/?$", re.I)
ARTICLE_RE = re.compile(r"/misq/vol(\d+)/iss(\d+)/(\d+)/?$", re.I)


def clean_text(s):
    if not s:
        return "N/A"
    return re.sub(r"\s+", " ", str(s)).strip()


def normalize_url(href):
    href = (href or "").strip()
    if not href:
        return ""
    if href.startswith("http://") or href.startswith("https://"):
        return href
    return urljoin(BASE_URL, href)


def get_soup(url, session):
    r = session.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")


def extract_issue_links(all_issues_soup):
    issue_links = []
    current_year = None

    for a in all_issues_soup.select("a[href]"):
        text = clean_text(a.get_text())
        href = normalize_url(a.get("href"))

        year_match = re.search(r"\((19\d{2}|20\d{2})\)", text)
        if year_match and "volume" in text.lower():
            current_year = int(year_match.group(1))
            continue

        if current_year is not None and current_year >= START_YEAR:
            if ISSUE_RE.search(href):
                issue_links.append((href.rstrip("/") + "/", current_year))

    seen = set()
    out = []
    for url, year in issue_links:
        if url not in seen:
            seen.add(url)
            out.append((url, year))

    return out


def extract_articles_only_from_issue(issue_url, year, issue_soup):
    rows = []

    m = ISSUE_RE.search(issue_url)
    if m:
        vol_issue = f"Vol {m.group(1)} Issue {m.group(2)}"
    else:
        vol_issue = "N/A"

    # Find the "Articles" heading
    articles_header = None
    for tag in issue_soup.find_all(["h2", "h3", "strong"]):
        if clean_text(tag.get_text()).lower() == "articles":
            articles_header = tag
            break

    if not articles_header:
        return rows

    # Walk forward until the next major section header
    node = articles_header.find_next_sibling()

    seen_urls = set()

    while node:
        # Stop when next section begins
        if node.name in ["h2", "h3"]:
            break

        # Collect only article title links, not PDFs
        for a in node.find_all("a", href=True):
            href = normalize_url(a["href"])

            if href.lower().endswith(".pdf"):
                continue

            if not ARTICLE_RE.search(href):
                continue

            href = href.rstrip("/") + "/"
            title = clean_text(a.get_text())

            if title == "N/A" or len(title) < 5:
                continue

            if href in seen_urls:
                continue
            seen_urls.add(href)

            rows.append([
                title,
                href,
                vol_issue,
                str(year)
            ])

        node = node.find_next_sibling()

    return rows


def write_rows(rows):
    file_exists = os.path.exists(OUT_FILE) and os.path.getsize(OUT_FILE) > 0

    with open(OUT_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        if not file_exists:
            writer.writerow([
                "Title",
                "URL",
                "Volume Issue",
                "Vol Issue Year"
            ])

        writer.writerows(rows)


def main():
    if os.path.exists(OUT_FILE):
        os.remove(OUT_FILE)

    total_rows = 0

    with requests.Session() as session:
        print("Loading:", START_URL)
        all_issues_soup = get_soup(START_URL, session)

        issue_links = extract_issue_links(all_issues_soup)
        print(f"Total issue pages found from {START_YEAR} to present: {len(issue_links)}")

        for idx, (issue_url, year) in enumerate(issue_links, start=1):
            print(f"[{idx}/{len(issue_links)}] {issue_url} | Year: {year}")

            try:
                issue_soup = get_soup(issue_url, session)
                rows = extract_articles_only_from_issue(issue_url, year, issue_soup)

                if rows:
                    write_rows(rows)
                    total_rows += len(rows)
                    print(f"   Saved {len(rows)} rows")
                else:
                    print("   No article rows found")

                time.sleep(random.uniform(0.5, 1.2))

            except Exception as e:
                print("   ERROR:", e)

    print("\nDone")
    print("Total rows saved:", total_rows)
    print("Output:", OUT_FILE)


if __name__ == "__main__":
    main()

## STEP 2: Collect article data from each article URL

In [6]:
import pandas as pd
import requests
import csv
import os
import time
import random
import re
from bs4 import BeautifulSoup

INPUT_FILE = "MISQ_Issues.csv"
OUT_FILE = "MISQ_article_data.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

JOURNAL_TITLE = "MIS Quarterly"


def clean_text(text):
    if not text:
        return "N/A"
    text = re.sub(r"\s+", " ", str(text).strip())
    return text if text else "N/A"


def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")


def extract_month_year(soup, fallback_year):
    text = soup.get_text(" ", strip=True)
    m = re.search(
        r"(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4}",
        text,
        re.I
    )
    if m:
        return clean_text(m.group(0))
    return str(fallback_year)


def extract_article_title(soup):
    h1_tags = soup.find_all("h1")
    if not h1_tags:
        return "N/A"

    # AIS page may contain more than one h1-like heading; choose the longest meaningful one
    candidates = [clean_text(h.get_text(" ", strip=True)) for h in h1_tags]
    candidates = [c for c in candidates if c != "N/A" and len(c) > 10]

    if not candidates:
        return "N/A"

    # real article title is usually the longest h1
    return max(candidates, key=len)


def extract_abstract(soup):
    abstract_heading = None

    for tag in soup.find_all(["h2", "h3", "strong"]):
        txt = clean_text(tag.get_text(" ", strip=True)).lower()
        if txt == "abstract":
            abstract_heading = tag
            break

    if not abstract_heading:
        return "N/A"

    texts = []
    node = abstract_heading.find_next_sibling()

    while node:
        if node.name in ["h2", "h3"]:
            break

        txt = clean_text(node.get_text(" ", strip=True))
        if txt != "N/A":
            texts.append(txt)

        node = node.find_next_sibling()

    return clean_text(" ".join(texts)) if texts else "N/A"


def extract_keywords(soup):
    text = soup.get_text(" ", strip=True)

    m = re.search(r"Keywords?\s*[:\-]?\s*(.+?)(?:Abstract|Download|$)", text, re.I)
    if m:
        kw = clean_text(m.group(1))
        if kw != "N/A" and len(kw) < 400:
            return kw

    meta_kw = soup.find("meta", attrs={"name": "keywords"})
    if meta_kw and meta_kw.get("content"):
        return clean_text(meta_kw["content"])

    return "N/A"


def extract_authors(soup, article_title):
    """Extract one row per author using on-page name + university text.

    Primary strategy: parse the explicit "Authors" block on the page;
    fallback strategy: look for elements with 'author' in their class
    names and nearby text between title and Abstract.
    """
    authors = []

    # ---------- 1) Try explicit "Authors" heading ----------
    blocks = []

    authors_heading = None
    for tag in soup.find_all(["h2", "h3", "strong"]):
        if clean_text(tag.get_text(" ", strip=True)).lower() == "authors":
            authors_heading = tag
            break

    if authors_heading:
        node = authors_heading.find_next_sibling()
        while node:
            # Stop when we hit the next major section, e.g., Abstract
            if node.name in ["h2", "h3", "strong"]:
                txt_heading = clean_text(node.get_text(" ", strip=True)).lower()
                if txt_heading in {"abstract", "keywords", "recommended citation"}:
                    break

            txt = clean_text(node.get_text(" ", strip=True))
            if txt != "N/A" and txt not in blocks:
                blocks.append(txt)

            node = node.find_next_sibling()

    # ---------- 2) Fallback: elements with 'author' in class ----------
    if not blocks:
        for tag in soup.find_all(True):
            classes = tag.get("class") or []
            if any("author" in str(c).lower() for c in classes):
                txt = clean_text(tag.get_text(" ", strip=True))
                if txt != "N/A" and txt not in blocks:
                    blocks.append(txt)

    # ---------- 3) Fallback: text between title <h1> and Abstract ----------
    if not blocks:
        title_node = None
        for h in soup.find_all("h1"):
            if clean_text(h.get_text(" ", strip=True)) == article_title:
                title_node = h
                break

        if title_node:
            abstract_heading = None
            for tag in soup.find_all(["h2", "h3", "strong"]):
                if clean_text(tag.get_text(" ", strip=True)).lower() == "abstract":
                    abstract_heading = tag
                    break

            node = title_node.find_next_sibling()
            while node:
                if abstract_heading and node == abstract_heading:
                    break

                txt = clean_text(node.get_text(" ", strip=True))
                if txt != "N/A" and txt not in blocks:
                    blocks.append(txt)

                node = node.find_next_sibling()

    if not blocks:
        return [{"name": "N/A", "email": "N/A", "address": "N/A"}]

    seen = set()

    for block in blocks:
        # Split possible multiple authors inside one block on 'Follow' and newlines
        segments = re.split(r"\bFollow\b|\n", block, flags=re.I)

        for seg in segments:
            seg = clean_text(seg)
            if seg == "N/A":
                continue

            if seg == article_title:
                continue
            if seg.lower() == JOURNAL_TITLE.lower():
                continue
            if seg.lower() in {"abstract", "authors"}:
                continue

            # Expect "Name , Affiliation" or "Name, Affiliation"
            if " , " in seg:
                name, addr = seg.split(" , ", 1)
            elif "," in seg:
                name, addr = seg.split(",", 1)
            else:
                # If we cannot split, treat as name only and skip
                continue

            name = clean_text(name)
            addr = clean_text(addr)

            if name == "N/A" or len(name) < 3:
                continue
            if name.lower() == JOURNAL_TITLE.lower():
                continue
            if name == article_title:
                continue

            addr = clean_text(addr)

            key = (name, addr)
            if key in seen:
                continue
            seen.add(key)

            authors.append({
                "name": name,
                "email": "N/A",  # not reliably available on-page
                "address": addr,
            })

    if not authors:
        authors = [{"name": "N/A", "email": "N/A", "address": "N/A"}]

    return authors


def extract_article_data(url, fallback_year):
    soup = get_soup(url)

    article_title = extract_article_title(soup)
    month_year = extract_month_year(soup, fallback_year)
    abstract = extract_abstract(soup)
    keywords = extract_keywords(soup)
    authors = extract_authors(soup, article_title)

    return article_title, month_year, abstract, keywords, authors


def write_rows(rows):
    file_exists = os.path.exists(OUT_FILE)

    with open(OUT_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        if not file_exists:
            writer.writerow([
                "URL",
                "Journal_Title",
                "Article_Title",
                "Volume_Issue",
                "Month_Year",
                "Abstract",
                "Keywords",
                "Author_name",
                "Author_email",
                "Author_Address"
            ])

        writer.writerows(rows)


def main():
    if os.path.exists(OUT_FILE):
        os.remove(OUT_FILE)

    df = pd.read_csv(INPUT_FILE)
    total = len(df)

    print("Total article URLs:", total)

    for i, row in df.iterrows():
        url = row["URL"]
        volume_issue = row["Volume Issue"]
        fallback_year = row["Vol Issue Year"]

        print(f"[{i+1}/{total}] {url}")

        try:
            article_title, month_year, abstract, keywords, authors = extract_article_data(url, fallback_year)

            rows = []
            for author in authors:
                rows.append([
                    url,
                    JOURNAL_TITLE,
                    article_title,
                    volume_issue,
                    month_year,
                    abstract,
                    keywords,
                    author["name"],
                    author["email"],
                    author["address"]
                ])

            write_rows(rows)
            print(f"   saved {len(rows)} author rows")

            time.sleep(random.uniform(0.5, 1.2))

        except Exception as e:
            print("   ERROR:", e)


if __name__ == "__main__":
    main()

Total article URLs: 938
[1/938] https://aisel.aisnet.org/misq/vol50/iss1/5/
   saved 19 author rows
[2/938] https://aisel.aisnet.org/misq/vol50/iss1/6/
   saved 4 author rows
[3/938] https://aisel.aisnet.org/misq/vol50/iss1/7/
   saved 7 author rows
[4/938] https://aisel.aisnet.org/misq/vol50/iss1/8/
   saved 3 author rows
[5/938] https://aisel.aisnet.org/misq/vol50/iss1/9/
   saved 3 author rows
[6/938] https://aisel.aisnet.org/misq/vol50/iss1/10/
   saved 3 author rows
[7/938] https://aisel.aisnet.org/misq/vol50/iss1/11/
   saved 4 author rows
[8/938] https://aisel.aisnet.org/misq/vol50/iss1/12/
   saved 4 author rows
[9/938] https://aisel.aisnet.org/misq/vol50/iss1/13/
   saved 4 author rows
[10/938] https://aisel.aisnet.org/misq/vol50/iss1/14/
   saved 4 author rows
[11/938] https://aisel.aisnet.org/misq/vol50/iss1/15/
   saved 4 author rows
[12/938] https://aisel.aisnet.org/misq/vol50/iss1/16/
   saved 5 author rows
[13/938] https://aisel.aisnet.org/misq/vol50/iss1/17/
   saved 3 